# Aula 1 — Um pedido, três papéis com LLM

Vamos passar o pedido P001 por estoque, logística e revisão. Ana, Bruno e Carla usam o mesmo modelo, cada um com seu prompt e seu histórico. Python define a ordem das etapas; o modelo gera as respostas.

A multiplicação continua em uma função pequena, como no Agente matemático. O modelo chama essa ferramenta, e nós conferimos o valor que ela devolveu. Todas as propostas da loja são fictícias.

## Preparar o Colab

Execute a célula e espere a instalação terminar. Depois, siga para o passo 1. As versões são fixas e vêm do PyPI oficial. Se o próprio Colab pedir reinício, use **Runtime → Restart session** e execute desde o início. Use uma sessão CPU; o modelo será acessado na nuvem do Ollama.

In [ ]:
import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError
IN_COLAB = 'google.colab' in sys.modules or bool(os.environ.get('COLAB_RELEASE_TAG'))
REQUISITOS = {'langchain': '1.4.7', 'langchain-core': '1.7.1', 'pydantic': '2.13.5', 'langchain-ollama': '1.1.0', 'langgraph': '1.2.18', 'ollama': '0.6.3'}
def versao_instalada(nome):
    try: return version(nome)
    except PackageNotFoundError: return None
faltantes = [f'{p}=={v}' for p, v in REQUISITOS.items() if versao_instalada(p) != v]
if IN_COLAB and faltantes:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--only-binary=:all:', '--disable-pip-version-check', '--index-url', 'https://pypi.org/simple', *faltantes], check=True)
elif faltantes: raise RuntimeError('Fora do Colab não instalamos automaticamente. Use o ambiente preparado nas versões indicadas no README.')
print('Python do kernel:', sys.executable, '| Colab:', IN_COLAB)
print('Versões conferidas:', {p: versao_instalada(p) for p in REQUISITOS})

## Antes de começar

No cadeado do Colab, confira o segredo chamado **segredo**, com sua chave do Ollama. Autorize o acesso deste notebook. A chave não deve ser colada nas células nem aparecer nas saídas.

Retomamos o P001 do Módulo 3: duas camisetas a R$ 35 e estoque inicial de oito. Os dados estão preparados neste caderno. Não há pagamento, reserva ou envio real.

## 1. Conectar o modelo

Esta célula importa o LangChain, lê o segredo e configura o Ollama. A primeira pergunta será enviada no passo 3. Não é necessário instalar o Ollama no computador nem baixar o modelo.

In [2]:
import os, json, requests
os.environ['LANGSMITH_TRACING'] = 'false'
os.environ['LANGCHAIN_TRACING_V2'] = 'false'
os.environ['OTEL_SDK_DISABLED'] = 'true'
import dotenv
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.messages import ToolMessage
# Local: Ollama na máquina, configurado pelo keys.env da raiz (sem chave de API)
dotenv.load_dotenv(dotenv.find_dotenv('keys.env', usecwd=True))
# temperature=0: o qwen2.5:3b segue regras e chama ferramentas com mais consistência
modelo = ChatOllama(model=os.getenv('OLLAMA_MODEL', 'qwen2.5:3b'),
    base_url=os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434'),
    temperature=0, num_predict=256, validate_model_on_init=False,
    client_kwargs={'timeout': 60})
print('Modelo configurado:', modelo.model, 'em', modelo.base_url)

Modelo configurado: qwen2.5:3b em http://localhost:11434


## Sua vez

Tente criar um sistema multiagente próprio. Utilize alguma das arquiteturas ensinadas.

In [3]:
# Cozinha do Creiton com coordenador: o coordenador recebe toda mensagem (do cliente ou de um agente),
# decide quem é o próximo e cria o envelope para ele. Os agentes só abrem envelopes endereçados a eles.
# Rode uma vez; o atendimento é o loop da próxima célula.
gramas_por_pessoa = 150
estoque_gramas = 1000

# valores devolvidos pelas tools dos agentes, guardados só para a conferência (o fluxo usa o texto dos agentes)
estado = {'pedido': None, 'total': None, 'sobra': None}

# Tools dos agentes: as contas ficam em Python, os agentes só pedem a chamada
@tool
def formata_pedido(mesa: str, prato: str, pessoas: int):
    """Formata os pedidos de cada mesa.
       mesa: Mesa do pedido (MXX)
       prato: Prato pedido pela mesa
       pessoas: quantidade de pessoas da mesa
    """
    pedido = {'mesa': mesa, 'prato': prato, 'pessoas': pessoas}
    estado['pedido'] = pedido
    return pedido

# gramas_por_pessoa e estoque_gramas ficam dentro das tools: cada agente só passa o número que recebeu na mensagem
@tool
def calcular_porcoes(pessoas: int) -> int:
    """Calcula o total de gramas necessário para servir todas as pessoas."""
    estado['total'] = pessoas * gramas_por_pessoa
    return estado['total']

@tool
def calcular_estoque(necessario: float) -> float:
    """Calcula quantos gramas sobram no estoque depois de separar o necessário. Resultado negativo indica falta."""
    estado['sobra'] = estoque_gramas - necessario
    return estado['sobra']

# Tools do correio: criar e entregar envelopes
@tool
def criar_envelope(remetente: str, destinatario: str, conteudo: str) -> dict:
    """Cria um envelope com quem envia, para quem vai e a mensagem."""
    return {'remetente': remetente, 'destinatario': destinatario, 'conteudo': conteudo}

@tool
def entregar_envelope(envelope: dict) -> dict:
    """Entrega o envelope: só o agente cujo nome está no destinatário abre, responde e devolve a resposta ao coordenador."""
    for nome, agente in agentes.items():
        if envelope['destinatario'] != nome:
            continue  # envelope de outro agente: não abre
        resposta = agente.invoke({'messages': [{'role': 'user', 'content': envelope['conteudo']}]}, config={'recursion_limit': 8})
        return criar_envelope.invoke({'remetente': nome, 'destinatario': 'COORDENADOR',
                                      'conteudo': resposta['messages'][-1].content})
    return envelope  # destinatário fora da cozinha (o cliente): o envelope volta como está

@tool
def enviar_envelope(destinatario: str, conteudo: str) -> str:
    """Cria um envelope do coordenador para o destinatário, entrega e devolve o envelope enviado e a resposta."""
    envelope = criar_envelope.invoke({'remetente': 'COORDENADOR', 'destinatario': destinatario, 'conteudo': conteudo})
    resposta = entregar_envelope.invoke({'envelope': envelope})
    return json.dumps({'enviado': envelope, 'resposta': resposta}, ensure_ascii=False)

# Tools do coordenador: uma por destinatário; a descrição diz que tipo de mensagem cada um recebe.
# Com uma tool só (destinatario como argumento), o qwen2.5:3b errava quase todas as rotas; com uma por destinatário, acertou 10/10.
# return_direct encerra a vez do coordenador assim que ele envia, e a resposta volta para ele no loop da próxima célula.
# "Estoque insuficiente" e "Pedido encerrado" foram escolhidos em teste: com "FALTA_INGREDIENTE" e "REPROVADO",
# o 3b mandava essas mensagens de volta para a Dona Rosa e para o Paulo.
@tool(return_direct=True)
def enviar_para_carlos(conteudo: str) -> str:
    """Cria um envelope para CARLOS, o garçom, e envia. Use para um pedido novo de cliente (texto livre) ou para uma mensagem que começa com APROVADO. conteudo: a mensagem a ser enviada."""
    return enviar_envelope.invoke({'destinatario': 'CARLOS', 'conteudo': conteudo})

@tool(return_direct=True)
def enviar_para_creiton(conteudo: str) -> str:
    """Cria um envelope para CREITON, o chef, e envia. Use para uma mensagem que começa com 'Pedido anotado'. conteudo: a mensagem a ser enviada."""
    return enviar_envelope.invoke({'destinatario': 'CREITON', 'conteudo': conteudo})

@tool(return_direct=True)
def enviar_para_dona_rosa(conteudo: str) -> str:
    """Cria um envelope para DONA ROSA, da despensa, e envia. Use para uma mensagem que começa com 'Total'. conteudo: a mensagem a ser enviada."""
    return enviar_envelope.invoke({'destinatario': 'DONA ROSA', 'conteudo': conteudo})

@tool(return_direct=True)
def enviar_para_paulo(conteudo: str) -> str:
    """Cria um envelope para PAULO, o revisor, e envia. Use para uma mensagem que começa com 'Proposta' ou 'Estoque insuficiente'. conteudo: a mensagem a ser enviada."""
    return enviar_envelope.invoke({'destinatario': 'PAULO', 'conteudo': conteudo})

@tool(return_direct=True)
def responder_cliente(conteudo: str) -> str:
    """Cria um envelope com a resposta final ao CLIENTE. Use para uma mensagem que começa com 'Entregue' ou 'Pedido encerrado'. conteudo: a mensagem a ser enviada."""
    return enviar_envelope.invoke({'destinatario': 'CLIENTE', 'conteudo': conteudo})

# Agentes: cada um responde só com a frase do seu modelo, que é o que o coordenador lê para decidir o próximo passo
coordenador = create_agent(modelo, tools=[enviar_para_carlos, enviar_para_creiton, enviar_para_dona_rosa,
                                          enviar_para_paulo, responder_cliente], system_prompt=(
    'Você é o coordenador do restaurante. Toda mensagem passa por você. Para cada mensagem recebida, crie exatamente um envelope '
    'para o próximo responsável chamando uma ferramenta, com a mensagem recebida copiada no conteúdo. Não escreva texto.'))
agentes = {
    # a regra do APROVADO vem primeiro e separada: como exceção no fim, o 3b anotava o APROVADO como pedido novo
    'CARLOS': create_agent(modelo, tools=[formata_pedido], system_prompt=(
        'Você é Carlos, o garçom do restaurante. Você recebe dois tipos de mensagem.\n'
        '1. Mensagem que começa com APROVADO: é uma entrega. NÃO chame ferramentas. Responda apenas "Entregue: <prato> na mesa <mesa>."\n'
        '2. Qualquer outra mensagem: é um pedido novo. Sua primeira ação é obrigatoriamente chamar formata_pedido com mesa no formato MXX, '
        'prato e pessoas. Depois do resultado, responda apenas "Pedido anotado: <mesa>, <prato>, <pessoas> pessoas."\n'
        'Responda em português, sem nenhum texto além da frase pedida.')),
    'CREITON': create_agent(modelo, tools=[calcular_porcoes], system_prompt=(
        'Você é Creiton, chef de cozinha. Você recebe um pedido anotado. '
        'Sempre chame calcular_porcoes com o número de pessoas antes de responder. '
        'Depois do resultado, responda apenas "Total: <resultado> g de <prato> para a mesa <mesa>." '
        'Responda em português, sem nenhum texto além da frase pedida.')),
    # formato de resposta fixo: sem ele o 3b confunde o que separar com o que sobra
    'DONA ROSA': create_agent(modelo, tools=[calcular_estoque], system_prompt=(
        'Você é Dona Rosa, da despensa. Você recebe o total de gramas de um prato. '
        'Sempre chame calcular_estoque com necessario=<total> antes de responder. A ferramenta devolve o que sobra no estoque. '
        'Se o resultado for negativo, responda apenas "Estoque insuficiente: faltam <resultado sem o sinal> g de <prato> para a mesa <mesa>." '
        'Se for zero ou positivo, responda apenas "Proposta: separar <total> g de <prato> para a mesa <mesa>; sobram <resultado> g no estoque." '
        'Responda em português, sem nenhum texto além da frase pedida.')),
    'PAULO': create_agent(modelo, tools=[], system_prompt=(
        'Você é Paulo, revisor. Você recebe o parecer da despensa. '
        'Se o parecer começar com Proposta, responda apenas "APROVADO: entregar <prato> na mesa <mesa>." '
        'Se o parecer começar com Estoque insuficiente, responda apenas "Pedido encerrado: o <prato> da mesa <mesa> não será servido hoje." '
        'Responda em português, sem nenhum texto além da frase pedida.')),
}

@tool
def conferir_atendimento(resposta_final: str) -> str:
    """Confere o atendimento pelos valores que as tools devolveram, não pelo texto dos agentes."""
    if None in estado.values():
        return 'PROBLEMA — alguma etapa não passou pela ferramenta (pedido, total ou estoque)'
    problemas = []
    if estado['total'] != estado['pedido']['pessoas'] * gramas_por_pessoa:
        problemas.append('o total não confere')
    if estado['sobra'] != estoque_gramas - estado['total']:
        problemas.append('a sobra do estoque não confere')
    if estado['sobra'] >= 0 and not resposta_final.startswith('Entregue'):
        problemas.append('com estoque, o pedido deveria ter sido entregue')
    if estado['sobra'] < 0 and not resposta_final.startswith('Pedido encerrado'):
        problemas.append('sem estoque, o cliente deveria ser avisado de que o pedido foi encerrado')
    return 'ok' if not problemas else 'PROBLEMA — ' + '; '.join(problemas)

print('Cozinha pronta: rode a próxima célula para fazer pedidos.')

Cozinha pronta: rode a próxima célula para fazer pedidos.


In [4]:
# Atendimento: o coordenador recebe cada mensagem e decide para quem criar o próximo envelope.
# Um pedido novo só é pedido quando o anterior volta ao cliente. Enter vazio encerra.
MAX_ENVELOPES = 12  # por pedido, para um erro de rota não deixar o loop girando para sempre
NOMES = {'CLIENTE': 'Cliente', 'COORDENADOR': 'Coordenador', 'CARLOS': 'Carlos', 'CREITON': 'Creiton',
         'DONA ROSA': 'Dona Rosa', 'PAULO': 'Paulo'}
recebido = None  # último envelope que chegou ao coordenador; None quando o pedido anterior terminou
while True:
    if recebido is None:
        pedido_cliente = input('Pedido da mesa (Enter vazio para sair): ').strip()
        if not pedido_cliente:
            break
        estado.update(pedido=None, total=None, sobra=None)
        recebido = criar_envelope.invoke({'remetente': 'CLIENTE', 'destinatario': 'COORDENADOR', 'conteudo': pedido_cliente})
        envelopes = 0
        print(f'\n_Cliente: {pedido_cliente}')

    decisao = coordenador.invoke({'messages': [{'role': 'user', 'content': recebido['conteudo']}]}, config={'recursion_limit': 6})
    envios = [json.loads(m.content) for m in decisao['messages'] if isinstance(m, ToolMessage)]
    if not envios:
        print('_Coordenador: não criou envelope. Resposta dele:', decisao['messages'][-1].content)
        recebido = None
        continue
    enviado, recebido = envios[-1]['enviado'], envios[-1]['resposta']
    envelopes += 1
    print(f"_Coordenador: envia para {NOMES[enviado['destinatario']]}")

    if enviado['destinatario'] == 'CLIENTE':
        print('Resposta final ao cliente:', enviado['conteudo'])
        print('Ferramentas:', estado)
        print('Conferência:', conferir_atendimento.invoke({'resposta_final': enviado['conteudo']}))
        recebido = None
    else:
        print(f"_{NOMES[recebido['remetente']]}: {recebido['conteudo']}")
        if envelopes >= MAX_ENVELOPES:
            print(f'Atendimento interrompido: o pedido passou de {MAX_ENVELOPES} envelopes sem voltar ao cliente.')
            recebido = None


_Cliente: mesaa 40 tão querendo vatapa de camarão pra 5 pessoas
_Coordenador: envia para Carlos
_Carlos: Pedido anotado: M40, vatapa de camarão, 5 pessoas.
_Coordenador: envia para Creiton
_Creiton: Total: 750 g de vatapa de camarão para a mesa M40.
_Coordenador: envia para Dona Rosa
_Dona Rosa: Proposta: separar 750 g de vatapa de camarão para a mesa M40; sobram 250.0 g no estoque.
_Coordenador: envia para Paulo
_Paulo: APROVADO: entregar 750 g de vatapa de camarão na mesa M40.
_Coordenador: envia para Carlos
_Carlos: Entregue: Vatapa de camarão na mesa M40.
_Coordenador: envia para Cliente
Resposta final ao cliente: Entregue: Vatapa de camarão na mesa M40.
Ferramentas: {'pedido': {'mesa': 'M40', 'prato': 'vatapa de camarão', 'pessoas': 5}, 'total': 750, 'sobra': 250.0}
Conferência: ok

_Cliente: mesa 2 querendo vatapa de frango pra 20 pessoas
_Coordenador: envia para Carlos
_Carlos: Pedido anotado: M02, vatapa de frango, 20 pessoas.
_Coordenador: envia para Carlos
_Carlos: Entregue:

## Para consultar

- [Agentes com create_agent — LangChain](https://docs.langchain.com/oss/python/langchain/agents)
- [ChatOllama — LangChain](https://docs.langchain.com/oss/python/integrations/chat/ollama)
- [API na nuvem — Ollama](https://docs.ollama.com/cloud)

Fontes conferidas em 29/10/2026. A execução depende da chave, do acesso ao modelo e da disponibilidade do Ollama.